# 02 - Silver Refining

Este notebook lê os dados brutos da camada Bronze, aplica regras de Data Quality (tratamento de nulos, correções de schema, padronização de formatos e flattening de JSON) e salva as tabelas limpas na camada Silver.

In [0]:
%run ./utils

# Utils

Notebook contendo funções reutilizáveis para o pipeline de dados.

In [0]:
%run ./data_quality

# Data Quality Framework

Este notebook centraliza as regras de validação, premissas de limpeza e definições de esquemas (Data Contracts) utilizados em todo o pipeline, garantindo consistência e transparência sobre a qualidade dos dados.

## 1. Definições de Esquemas (Schema Enforcement)

Utilizamos `StructType` do Spark para forçar a tipagem correta na camada Silver, garantindo que mudanças na fonte não quebrem o consumo analítico.

## 2. Dicionários de Referência

## 3. Funções de Validação e Padronização

## 4. Premissas e Limitações Adotadas (Documentação Técnica)

### Geral
- **Padronização de Nulos**: Todos os campos de texto categóricos nulos ou vazios são consolidados no termo único **`NAO INFORMADO`**. Isso evita que registros sejam excluídos acidentalmente em filtros de BI.
- **Schema Enforcement**: Escolhemos utilizar `StructType` do Spark em vez de bibliotecas como Pydantic para priorizar a performance em larga escala, evitando custos de serialização entre Spark e Python.

### Clientes
- **Geografia**: Aplicamos uma normalização defensiva que converte variações de nomes de estados para siglas oficiais (UFs). Estados não reconhecidos são marcados como `NAO INFORMADO`.
- **Status**: Tratamento binário onde apenas 'ativo' é considerado como verdadeiro para a flag (`flg_ativo = True`).

### Produtos
- **Preços**: Produtos com `list_price` zerado são sinalizados via `flg_qualidade`, permitindo que o analista decida se deve incluí-los em cálculos financeiros.

### Pedidos
- **Itens (Status)**: Definimos que itens de pedido com status nulo ou em branco na origem são automaticamente classificados como **CANCELADO**. Valores preenchidos são padronizados para maiúsculo.
- **Prioridade**: Definimos a premissa de que registros com `priority` nulo no JSON de origem são classificados como **low**, refletindo o comportamento padrão do sistema de pedidos quando não há urgência sinalizada.
- **Inconsistência Numérica**: O ERP apresenta mistura de separadores decimais (vírgula e ponto). Ajuste via Regex para garantir a integridade dos cálculos de receita.
- **Integridade de Valores**: Marcamos como baixa qualidade (`flg_qualidade = False`) pedidos onde o valor líquido é superior ao valor bruto, o que indicaria um erro de cálculo de impostos/descontos.
- **Duplicidade de Itens**: Registros com o mesmo `order_id` que possuam `item_seq` ou `product_code` repetidos são sinalizados como baixa qualidade para evitar distorção em métricas de volume e SKU.


### Vendedores
- **Status**: Vendedores com status nulo ou em branco na origem são considerados **ativos** (`flg_ativo = True`).

### Regiões
- **Padronização Geográfica**: Regional e Estado são padronizados para siglas (ex: `S`, `SP`) para garantir unicidade e joins precisos. Registros duplicados após padronização são removidos.
### Performance
- **Z-Ordering**: Aplicamos Z-Order nas colunas de busca frequente para otimizar o Data Skipping do Delta Lake.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DateType, DecimalType, DoubleType, StructType, StructField, StringType

# Função auxiliar para ler da Bronze
def read_bronze_table(table_name):
    return spark.read.table(f"{catalog_name}.bronze.{table_name}")

### 1. Clientes

In [0]:
df_clientes = read_bronze_table("clientes_crm")

# --- TRATAMENTOS DE QUALIDADE ---
df_clientes = df_clientes.withColumn("data_cadastro_clean", parse_multi_date("data_cadastro"))

# Normalização de Estados
state_expr = F.when(F.lower(F.col("estado")).isin(list(STATE_MAPPING.keys())),
                    F.create_map([F.lit(x) for x in sum(STATE_MAPPING.items(), ())])[F.lower(F.col("estado"))])
state_expr = state_expr.when(F.upper(F.col("estado")).isin(VALID_UFS), F.upper(F.col("estado")))
state_expr = state_expr.otherwise(F.lit(DEFAULT_NULL_TEXT))

df_clientes = df_clientes.withColumn("estado_clean", state_expr)

# --- DATA QUALITY GATE ---
df_clientes = df_clientes.withColumn(
    "flg_qualidade",
    is_valid_state("estado_clean") &
    is_valid_email("email") &
    F.col("data_cadastro_clean").isNotNull()
)

# --- PROJEÇÃO E SCHEMA ENFORCEMENT ---
df_clientes_silver = df_clientes.select(
    F.col("customer_id"),
    standardize_null_text("nome_cliente").alias("nome_cliente"),
    standardize_null_text("segmento").alias("segmento"),
    standardize_null_text("porte").alias("porte"),
    standardize_null_text("cidade").alias("cidade"),
    F.col("estado_clean").alias("estado"),
    F.col("data_cadastro_clean").alias("data_cadastro"),
    standardize_null_text("email").alias("email"),
    F.when(F.lower(F.col("status_cliente")) == "ativo", True).otherwise(False).alias("flg_ativo"),
    F.col("flg_qualidade"),
    F.current_timestamp().alias("_silver_processed_at")
)

for field in SCHEMA_CLIENTES_SILVER:
    df_clientes_silver = df_clientes_silver.withColumn(field.name, F.col(field.name).cast(field.dataType))

save_delta_table(df_clientes_silver, "clientes", "silver", catalog=catalog_name)
optimize_table("clientes", "silver", zorder_cols=["customer_id", "nome_cliente", "estado"], catalog=catalog_name)


Salvando tabela gerenciada: workspace.silver.clientes no modo overwrite...
Salvo com sucesso!
Executando otimização: OPTIMIZE workspace.silver.clientes ZORDER BY (customer_id, nome_cliente, estado)...
Otimização finalizada com sucesso!


### 2. Produtos

In [0]:
watermark = "1900-01-01 00:00:00"
full_silver_table = f"{catalog_name}.silver.produtos"
if spark.catalog.tableExists(full_silver_table):
    watermark_val = spark.table(full_silver_table).select(F.max("updated_at")).collect()[0][0]
    if watermark_val: watermark = str(watermark_val)

df_produtos = read_bronze_table("produtos_dump").filter(F.col("updated_at") > watermark)

if df_produtos.count() > 0:
    df_produtos_silver = df_produtos.select(
        F.col("product.product_id").alias("product_id"),
        standardize_null_text("product.name").alias("product_name"),
        standardize_null_text("product.category").alias("category"),
        standardize_null_text("product.subcategory").alias("subcategory"),
        F.upper(standardize_null_text("product.status")).alias("status"),
        clean_numeric("pricing.list_price").alias("list_price"),
        standardize_null_text("pricing.currency").alias("currency"),
        standardize_null_text("attributes.family").alias("family"),
        standardize_null_text(F.concat_ws(",", F.col("attributes.tags"))).alias("tags"),
        parse_multi_timestamp("updated_at").alias("updated_at"),
        is_positive_number(clean_numeric("pricing.list_price")).alias("flg_qualidade"),
        F.current_timestamp().alias("_silver_processed_at")
    )
    
    # Aplicação dinâmica do Schema definido no data_quality.ipynb
    for field in SCHEMA_PRODUTOS_SILVER:
        df_produtos_silver = df_produtos_silver.withColumn(field.name, F.col(field.name).cast(field.dataType))
    
    upsert_delta_table(df_produtos_silver, "produtos", "silver", join_key="product_id", catalog=catalog_name)
    optimize_table("produtos", "silver", zorder_cols=["product_id", "product_name", "category", "subcategory", "status", "family"], catalog=catalog_name)
else:
    print("Nenhum produto novo para processar.")


Realizando MERGE na tabela: workspace.silver.produtos...
MERGE finalizado com sucesso!
Executando otimização: OPTIMIZE workspace.silver.produtos ZORDER BY (product_id, product_name, category, subcategory, status, family)...
Otimização finalizada com sucesso!


### 3. Pedidos Cabecalho

In [0]:
print("Processando Pedidos Cabeçalho (Silver Incremental com Watermark)...")

watermark = "1900-01-01 00:00:00"
full_silver_table = f"{catalog_name}.silver.pedidos"
if spark.catalog.tableExists(full_silver_table):
    # Buscamos a maior data de atualização já processada na Silver
    watermark_val = spark.table(full_silver_table).select(F.max("data_atualizacao")).collect()[0][0]
    if watermark_val: watermark = str(watermark_val)

df_pedidos = read_bronze_table("pedidos_cabecalho").filter(F.col("last_update") > watermark)

if df_pedidos.count() > 0:
    # 1. Parse do JSON de Payment Details
    json_schema = "priority STRING, source STRING"
    df_pedidos = df_pedidos.withColumn("payment_struct", F.from_json(F.col("payment_details"), json_schema))

    # 2. Limpeza de Datas e Números (com funções centralizadas)
    df_pedidos = df_pedidos.withColumn("order_date_clean", parse_multi_date("order_date")) \
                           .withColumn("promised_date_clean", parse_multi_date("promised_date")) \
                           .withColumn("gross_amount_clean", clean_numeric("gross_amount")) \
                           .withColumn("discount_amount_clean", clean_numeric("discount_amount")) \
                           .withColumn("net_amount_clean", clean_numeric("net_amount"))

    # 3. Lógica de Prioridade e Status
    df_pedidos = df_pedidos.withColumn("priority_clean", handle_priority("payment_struct.priority"))

    # --- DATA QUALITY GATE ---
    df_pedidos = df_pedidos.withColumn(
        "flg_qualidade",
        (F.col("order_date_clean").isNotNull()) &
        (F.col("promised_date_clean").isNotNull()) &
        is_positive_number("gross_amount_clean") &
        is_positive_number("discount_amount_clean") &
        is_positive_number("net_amount_clean") &
        (F.col("customer_code").isNotNull()) &
        (F.col("net_amount_clean") <= F.col("gross_amount_clean"))
    )

    # --- PROJEÇÃO E SCHEMA ENFORCEMENT ---
    df_pedidos_silver = df_pedidos.select(
        F.col("order_id").alias("pedido_id"),
        F.col("customer_code").alias("customer_id"),
        F.col("seller_id").alias("vendedor_id"),
        F.col("order_date_clean").alias("data_pedido"),
        F.col("promised_date_clean").alias("data_previsao_entrega"),
        standardize_null_text(F.upper(F.col("status_order"))).alias("status_pedido"),
        F.col("gross_amount_clean").alias("valor_bruto"),
        F.col("discount_amount_clean").alias("valor_desconto"),
        F.col("net_amount_clean").alias("valor_liquido"),
        standardize_null_text("priority_clean").alias("prioridade"),
        standardize_null_text("payment_struct.source").alias("origem_pedido"),
        parse_multi_timestamp("last_update").alias("data_atualizacao"),
        F.col("flg_qualidade"),
        F.current_timestamp().alias("_silver_processed_at")
    )

    for field in SCHEMA_PEDIDOS_SILVER:
        df_pedidos_silver = df_pedidos_silver.withColumn(field.name, F.col(field.name).cast(field.dataType))

    upsert_delta_table(df_pedidos_silver, "pedidos", "silver", join_key="pedido_id", catalog=catalog_name)
    optimize_table("pedidos", "silver", zorder_cols=["pedido_id", "customer_id", "data_pedido"], catalog=catalog_name)
else:
    print("Nenhum pedido novo para processar na Silver.")


Processando Pedidos Cabeçalho (Silver Incremental com Watermark)...
Nenhum pedido novo para processar na Silver.


### 4. Pedidos Itens

In [0]:
from pyspark.sql.window import Window
df_pedidos_itens = read_bronze_table("pedidos_itens")

# --- TRATAMENTOS DE QUALIDADE ---
df_pedidos_itens = df_pedidos_itens.withColumn("item_id_clean", clean_int("item_seq"))
df_pedidos_itens = df_pedidos_itens.withColumn("quantidade_clean", clean_int("quantity"))
df_pedidos_itens = df_pedidos_itens.withColumn("preco_unit_clean", clean_numeric("unit_price"))
df_pedidos_itens = df_pedidos_itens.withColumn("valor_total_clean", clean_numeric("total_item"))

# Window Functions para detectar duplicidade dentro do mesmo order_id
w_seq = Window.partitionBy("order_id", "item_seq")
w_prod = Window.partitionBy("order_id", "product_code")

df_pedidos_itens = df_pedidos_itens.withColumn("count_seq", F.count("*").over(w_seq))
df_pedidos_itens = df_pedidos_itens.withColumn("count_prod", F.count("*").over(w_prod))

# --- DATA QUALITY GATE ---
df_pedidos_itens = df_pedidos_itens.withColumn(
    "flg_qualidade",
    F.col("order_id").isNotNull() &
    (F.col("count_seq") == 1) &
    (F.col("count_prod") == 1) &
    is_positive_number("quantidade_clean") &
    is_positive_number("preco_unit_clean") &
    is_positive_number("valor_total_clean")
)

# --- PROJEÇÃO E SCHEMA ENFORCEMENT ---
df_itens_silver = df_pedidos_itens.select(
    F.col("order_id").alias("pedido_id"),
    F.col("item_id_clean").alias("item_id"),
    F.col("product_code").alias("product_id"),
    F.col("quantidade_clean").alias("quantidade"),
    F.col("preco_unit_clean").alias("preco_unitario"),
    F.col("valor_total_clean").alias("valor_item"),
    handle_item_status("item_status").alias("status_item"),
    F.col("flg_qualidade"),
    F.current_timestamp().alias("_silver_processed_at")
)

for field in SCHEMA_PEDIDOS_ITENS_SILVER:
    df_itens_silver = df_itens_silver.withColumn(field.name, F.col(field.name).cast(field.dataType))

save_delta_table(df_itens_silver, "pedidos_itens", "silver", catalog=catalog_name)
optimize_table("pedidos_itens", "silver", zorder_cols=["pedido_id", "product_id"], catalog=catalog_name)


Salvando tabela gerenciada: workspace.silver.pedidos_itens no modo overwrite...
Salvo com sucesso!
Executando otimização: OPTIMIZE workspace.silver.pedidos_itens ZORDER BY (pedido_id, product_id)...
Otimização finalizada com sucesso!


### 5. Regiões


In [0]:
df_regioes = read_bronze_table("regioes")

# Mapeamento para siglas de região

regional_expr = F.create_map([F.lit(x) for x in sum(REGIONAL_MAPPING.items(), ())])[F.lower(F.col("regional_code"))]
regional_expr = F.coalesce(regional_expr, F.upper(F.col("regional_code")))

# Mapeamento para siglas de estado (reutilizando STATE_MAPPING de Clientes)
state_expr = F.create_map([F.lit(x) for x in sum(STATE_MAPPING.items(), ())])[F.lower(F.col("state"))]
state_expr = F.coalesce(state_expr, F.upper(F.col("state")))

# --- DATA QUALITY GATE ---
df_regioes = df_regioes.withColumn(
    "flg_qualidade",
    F.col("regional_code").isNotNull() & F.col("state").isNotNull()
)

# --- PROJEÇÃO E SCHEMA ENFORCEMENT ---
df_regioes_silver = df_regioes.select(
    standardize_null_text(regional_expr).alias("regional_code"),
    standardize_null_text("regional_name").alias("regional_name"),
    standardize_null_text(state_expr).alias("state"),
    standardize_null_text("manager_name").alias("manager_name"),
    (F.col("active_flag") == 1).alias("flg_ativo"),
    F.col("flg_qualidade"),
    F.current_timestamp().alias("_silver_processed_at")
)

# DEDUPLICAÇÃO
df_regioes_silver = df_regioes_silver.dropDuplicates(["regional_code", "state"])

for field in SCHEMA_REGIOES_SILVER:
    df_regioes_silver = df_regioes_silver.withColumn(field.name, F.col(field.name).cast(field.dataType))

save_delta_table(df_regioes_silver, "regioes", "silver", catalog=catalog_name)
optimize_table("regioes", "silver", zorder_cols=["regional_code"], catalog=catalog_name)


Salvando tabela gerenciada: workspace.silver.regioes no modo overwrite...
Salvo com sucesso!
Executando otimização: OPTIMIZE workspace.silver.regioes ZORDER BY (regional_code)...
Otimização finalizada com sucesso!


### 6. Comercial Canais


In [0]:
df_canais = read_bronze_table("comercial_canais")

# --- DATA QUALITY GATE ---
df_canais = df_canais.withColumn(
    "flg_qualidade",
    F.col("id_canal").isNotNull()
)

# --- PROJEÇÃO E SCHEMA ENFORCEMENT ---
df_canais_silver = df_canais.select(
    F.col("id_canal").cast("string"),
    standardize_null_text("nome_canal").alias("nome_canal"),
    standardize_null_text("tipo_canal").alias("tipo_canal"),
    F.when(F.lower(F.col("ativo")) == "sim", True).otherwise(False).alias("flg_ativo"),
    standardize_null_text("observacao").alias("observacao"),
    F.col("flg_qualidade"),
    F.current_timestamp().alias("_silver_processed_at")
)

for field in SCHEMA_CANAIS_SILVER:
    df_canais_silver = df_canais_silver.withColumn(field.name, F.col(field.name).cast(field.dataType))
df_canais_silver = df_canais_silver.dropDuplicates(["id_canal"])
save_delta_table(df_canais_silver, "canais", "silver", catalog=catalog_name)
optimize_table("canais", "silver", zorder_cols=["id_canal", "nome_canal"], catalog=catalog_name)


Salvando tabela gerenciada: workspace.silver.canais no modo overwrite...
Salvo com sucesso!
Executando otimização: OPTIMIZE workspace.silver.canais ZORDER BY (id_canal, nome_canal)...
Otimização finalizada com sucesso!


### 7. Vendedores


In [0]:
print("Processando Vendedores (Silver)...")
df_vendedores = read_bronze_table("vendedores")

# Reutilizando lógica de regional de Regiões

regional_expr = F.create_map([F.lit(x) for x in sum(REGIONAL_MAPPING.items(), ())])[F.lower(F.col("regional_code"))]
regional_expr = F.coalesce(regional_expr, F.upper(F.col("regional_code")))

# --- TRATAMENTOS DE QUALIDADE ---
df_vendedores = df_vendedores.withColumn("hire_date_clean", parse_multi_date("hire_date"))

# --- DATA QUALITY GATE ---
df_vendedores = df_vendedores.withColumn(
    "flg_qualidade",
    F.col("seller_id").isNotNull() & F.col("hire_date_clean").isNotNull()
)

# --- PROJEÇÃO E SCHEMA ENFORCEMENT ---
df_vendedores_silver = df_vendedores.select(
    F.col("seller_id"),
    standardize_null_text("seller_name").alias("seller_name"),
    F.col("canal_id").cast("string"), # Ajustado para string pois id_canal em Canais é string
    standardize_null_text(regional_expr).alias("regional_code"),
    F.col("hire_date_clean").alias("hire_date"),
    (F.lower(F.col("status")).isin("ativo", "") | F.col("status").isNull()).alias("flg_ativo"),
    F.col("flg_qualidade"),
    F.current_timestamp().alias("_silver_processed_at")
)

for field in SCHEMA_VENDEDORES_SILVER:
    df_vendedores_silver = df_vendedores_silver.withColumn(field.name, F.col(field.name).cast(field.dataType))

save_delta_table(df_vendedores_silver, "vendedores", "silver", catalog=catalog_name)
optimize_table("vendedores", "silver", zorder_cols=["seller_id", "regional_code"], catalog=catalog_name)


Processando Vendedores (Silver)...
Salvando tabela gerenciada: workspace.silver.vendedores no modo overwrite...
Salvo com sucesso!
Executando otimização: OPTIMIZE workspace.silver.vendedores ZORDER BY (seller_id, regional_code)...
Otimização finalizada com sucesso!


### 8. Entregas


In [0]:
df_entregas = read_bronze_table("entregas")

# --- TRATAMENTOS DE QUALIDADE ---
df_entregas = df_entregas.withColumn("cost_clean", clean_numeric("cost"))

# --- DATA QUALITY GATE ---
df_entregas = df_entregas.withColumn(
    "flg_qualidade",
    F.col("order_ref").isNotNull() & is_positive_number("cost_clean")
)

# --- PROJEÇÃO E SCHEMA ENFORCEMENT ---
df_entregas_silver = df_entregas.select(
    F.col("delivery_id"),
    F.col("order_ref").alias("order_id"),
    standardize_null_text("carrier.name").alias("carrier_name"),
    standardize_null_text("carrier.mode").alias("carrier_mode"),
    standardize_null_text("delivery_status").alias("delivery_status"),
    parse_multi_timestamp("timestamps.shipped_at").alias("shipped_at"),
    parse_multi_timestamp("timestamps.delivered_at").alias("delivered_at"),
    standardize_null_text("destination.state").alias("destination_state"),
    standardize_null_text("destination.city").alias("destination_city"),
    F.col("cost_clean").alias("cost"),
    F.col("flg_qualidade"),
    F.current_timestamp().alias("_silver_processed_at")
)

for field in SCHEMA_ENTREGAS_SILVER:
    df_entregas_silver = df_entregas_silver.withColumn(field.name, F.col(field.name).cast(field.dataType))

save_delta_table(df_entregas_silver, "entregas", "silver", catalog=catalog_name)
optimize_table("entregas", "silver", zorder_cols=["order_id", "delivery_id"], catalog=catalog_name)


Salvando tabela gerenciada: workspace.silver.entregas no modo overwrite...
Salvo com sucesso!
Executando otimização: OPTIMIZE workspace.silver.entregas ZORDER BY (order_id, delivery_id)...
Otimização finalizada com sucesso!


### 9. Ocorrências


In [0]:
df_ocorrencias = read_bronze_table("ocorrencias")

# --- DATA QUALITY GATE ---
df_ocorrencias = df_ocorrencias.withColumn(
    "flg_qualidade",
    F.col("ticket_id").isNotNull() & F.col("order_id").isNotNull()
)

# --- PROJEÇÃO E SCHEMA ENFORCEMENT ---
df_ocorrencias_silver = df_ocorrencias.select(
    F.col("ticket_id"),
    F.col("order_id"),
    standardize_null_text("event_type").alias("event_type"),
    parse_multi_timestamp("created_at").alias("created_at"),
    standardize_null_text(F.upper(F.col("severity"))).alias("severity"),
    standardize_null_text(F.upper(F.col("status"))).alias("status"),
    F.col("flg_qualidade"),
    F.current_timestamp().alias("_silver_processed_at")
)

for field in SCHEMA_OCORRENCIAS_SILVER:
    df_ocorrencias_silver = df_ocorrencias_silver.withColumn(field.name, F.col(field.name).cast(field.dataType))

save_delta_table(df_ocorrencias_silver, "ocorrencias", "silver", catalog=catalog_name)
optimize_table("ocorrencias", "silver", zorder_cols=["order_id", "ticket_id"], catalog=catalog_name)

print("Processamento Silver 100% concluído com qualidade máxima aplicada!")

Salvando tabela gerenciada: workspace.silver.ocorrencias no modo overwrite...
Salvo com sucesso!
Executando otimização: OPTIMIZE workspace.silver.ocorrencias ZORDER BY (order_id, ticket_id)...
Otimização finalizada com sucesso!
Processamento Silver 100% concluído com qualidade máxima aplicada!
